# Knowledge Graphs and Semantic Technologies -- OWL tutorial

First install owlready2 if you don't already have it, and have a quick look at the [documentation](https://owlready2.readthedocs.io/en/v0.36/onto.html#).

In [1]:
## Uncomment if you do not have owlrl installed (you should have it installed from the RDFS tutorial)
#import sys
#!{sys.executable} -m pip install rdflib  owlready2 pandas

import pandas as pd
from rdflib import Graph, Literal, Namespace, RDF, URIRef, OWL
from rdflib.namespace import DC, FOAF

from owlready2 import *

Let's start loading some data from a .CSV file. We are going to create an ontology that describes the data inside.
We already did part of this using the semantics of RDF(S), now we'll use the semantics of [OWL](https://www.w3.org/TR/2012/REC-owl2-primer-20121211/) through owlready2. 

Remember that an ontology is often an application ontology, meaning that it is built with a specific task in mind. 
We could model _everything_ within a certain domain in the most ontologically correct way possible, _or_ **we could model the domain in accordance with the application's task.** 


**Your task and domain:** You are a broadcaster that has just digitised its radio archives into a digital music archive (DMA), and aims to play more interesting tracks by discovering their 'hidden treasures', by making unexpected and potentially interesting relations between tracks visible to the users (which are journalists and program makers).


**Exercise 1** 

1. look at the .csv files in the folder /data/musicoset_metadata/ and load them into pandas dataframes (use display.max_columns to show all columns). 
2. initialise an empty ontology using owlready2
3. using owlready2, create a hierarchy of classes and subclasses that describe the entities in your dataframes
4. using owrleady2, create properties and subproperties their properties, and how the classes relate to one another (using domain and range). If it helps: draw out your ontology in https://app.diagrams.net/
    - create: object properties, data properties, functional properties
5. using owlready2, add class restrictions
6. create invididuals of your classes, and provide them with attributes using your properties! 
7. write simple queries to retrieve your individuals following: https://owlready2.readthedocs.io/en/v0.36/onto.html#simple-queries. What kind of things would journalists and program makers like to retrieve? 
6. save your asserted owl file

In [2]:
#load each csv in a different df
pd.set_option("display.max_columns", None)

data_dir = "data/musicoset_metadata"
csv_paths = sorted([
    os.path.join(data_dir, f)
    for f in os.listdir(data_dir)
    if f.lower().endswith(".csv")
])

dfs = {}
failed = {}

for p in csv_paths:
    name = os.path.splitext(os.path.basename(p))[0]
    try:
        dfs[name] = pd.read_csv(p)
    except Exception as e1:
        try:
            dfs[name] = pd.read_csv(p, engine="python", on_bad_lines="skip")
        except Exception as e2:
            try:
                dfs[name] = pd.read_csv(p, sep=";", engine="python", on_bad_lines="skip")
            except Exception as e3:
                failed[name] = (str(e1), str(e2), str(e3))

list(dfs.keys()), failed

(['albums', 'artists', 'releases', 'songs', 'tracks'], {})

In [3]:
#Create classes - each class is subclass of thing
onto = get_ontology("http://example.org/dma_musicoset.owl")

with onto:
    class Track(Thing): pass
    class Recording(Thing): pass
    class Release(Thing): pass
    class Album(Release): pass
    class Single(Release): pass
    class Person(Thing): pass
    class Group(Thing): pass
    class Artist(Thing): pass
    class Composer(Person): pass
    class Performer(Artist): pass
    class Genre(Thing): pass
    class SubGenre(Genre): pass
    class Instrument(Thing): pass
    class Language(Thing): pass
    class Label(Thing): pass
    class Country(Thing): pass
    class Mood(Thing): pass
    class Collection(Thing): pass

In [4]:
#Create properties and subproperties
with onto:
    class hasRecording(ObjectProperty):
        domain = [Track]
        range = [Recording]

    class hasRelease(ObjectProperty):
        domain = [Track]
        range = [Release]

    class hasAlbum(ObjectProperty):
        domain = [Track]
        range = [Album]

    class hasArtist(ObjectProperty):
        domain = [Track]
        range = [Artist]

    class hasPerformer(ObjectProperty):
        domain = [Track]
        range = [Performer]

    class hasComposer(ObjectProperty):
        domain = [Track]
        range = [Composer]

    class hasGenre(ObjectProperty):
        domain = [Track]
        range = [Genre]

    class hasSubGenre(ObjectProperty):
        domain = [Track]
        range = [SubGenre]

    class hasInstrument(ObjectProperty):
        domain = [Track]
        range = [Instrument]

    class hasLanguage(ObjectProperty):
        domain = [Track]
        range = [Language]

    class hasLabel(ObjectProperty):
        domain = [Release]
        range = [Label]

    class hasCountry(ObjectProperty):
        domain = [Artist]
        range = [Country]

    class hasMood(ObjectProperty):
        domain = [Track]
        range = [Mood]

    class inCollection(ObjectProperty):
        domain = [Track]
        range = [Collection]

    class trackTitle(DataProperty, FunctionalProperty):
        domain = [Track]
        range = [str]

    class trackId(DataProperty, FunctionalProperty):
        domain = [Track]
        range = [str]

    class durationSeconds(DataProperty, FunctionalProperty):
        domain = [Track]
        range = [int]

    class releaseYear(DataProperty, FunctionalProperty):
        domain = [Release]
        range = [int]

    class releaseTitle(DataProperty, FunctionalProperty):
        domain = [Release]
        range = [str]

    class artistName(DataProperty, FunctionalProperty):
        domain = [Artist]
        range = [str]

    class entityName(DataProperty, FunctionalProperty):
        domain = [Thing]
        range = [str]

    class genreName(DataProperty, FunctionalProperty):
        domain = [Genre]
        range = [str]

    class instrumentName(DataProperty, FunctionalProperty):
        domain = [Instrument]
        range = [str]

    class languageName(DataProperty, FunctionalProperty):
        domain = [Language]
        range = [str]

    class moodName(DataProperty, FunctionalProperty):
        domain = [Mood]
        range = [str]

    class labelName(DataProperty, FunctionalProperty):
        domain = [Label]
        range = [str]

    class collectionName(DataProperty, FunctionalProperty):
        domain = [Collection]
        range = [str]

In [5]:
#add class restrictions
with onto:
    Track.is_a.append(hasArtist.some(Artist))
    Track.is_a.append(hasGenre.some(Genre))
    Track.is_a.append(trackTitle.exactly(1, str))
    Release.is_a.append(releaseTitle.max(1, str))
    Artist.is_a.append(artistName.max(1, str))

In [6]:
#create instances of the classes
def _slug(x):
    s = "" if x is None else str(x)
    s = s.strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = s.strip("_")
    return s if s else "unknown"

def _as_list(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return []
    if isinstance(v, (list, tuple, set)):
        return [str(x).strip() for x in v if str(x).strip()]
    s = str(v).strip()
    if not s or s.lower() in {"nan", "none"}:
        return []
    parts = re.split(r"\s*[,;/|]\s*", s)
    return [p.strip() for p in parts if p.strip()]

def _first_present(row, keys):
    for k in keys:
        if k in row and pd.notna(row[k]) and str(row[k]).strip():
            return row[k]
    return None

track_index = {}
artist_index = {}
genre_index = {}
subgenre_index = {}
instrument_index = {}
language_index = {}
mood_index = {}
release_index = {}
label_index = {}
country_index = {}
collection_index = {}

def _get_or_make(cls, idx, key, name_prop=None, name_value=None):
    if key in idx:
        return idx[key]
    ind = cls(key)
    if name_prop is not None and name_value is not None and str(name_value).strip():
        setattr(ind, name_prop, str(name_value).strip())
    idx[key] = ind
    return ind

def _get_track(row):
    tid = _first_present(row, ["track_id", "trackid", "id", "song_id", "recording_id", "spotify_track_id", "mb_track_id"])
    title = _first_present(row, ["track_name", "title", "name", "track", "song", "track_title"])
    if tid is None:
        tid = f"track_{_slug(title)}"
    key = f"track_{_slug(tid)}"
    t = _get_or_make(onto.Track, track_index, key)
    if title is not None and (not getattr(t, "trackTitle", None)):
        t.trackTitle = str(title).strip()
    if tid is not None and (not getattr(t, "trackId", None)):
        t.trackId = str(tid).strip()
    dur = _first_present(row, ["duration", "duration_sec", "duration_seconds", "length", "track_duration"])
    if dur is not None:
        try:
            t.durationSeconds = int(float(dur))
        except Exception:
            pass
    return t

def _attach_many(track, prop, cls, idx, values, name_prop):
    for v in values:
        key = f"{cls.__name__.lower()}_{_slug(v)}"
        ind = _get_or_make(cls, idx, key, name_prop=name_prop, name_value=v)
        getattr(track, prop).append(ind)

def _ensure_release_for_track(track, row):
    rtitle = _first_present(row, ["album", "album_name", "release", "release_name", "record", "collection", "release_title"])
    rtype = _first_present(row, ["release_type", "album_type", "type"])
    year = _first_present(row, ["year", "release_year", "date", "releaseDate"])
    if rtitle is None and year is None:
        return None
    base_key = f"release_{_slug(rtitle) if rtitle is not None else 'unknown'}_{_slug(year) if year is not None else 'unknown'}"
    if base_key in release_index:
        rel = release_index[base_key]
    else:
        rel_cls = onto.Release
        if rtype is not None:
            rt = str(rtype).strip().lower()
            if "album" in rt or "lp" in rt:
                rel_cls = onto.Album
            elif "single" in rt or "ep" in rt:
                rel_cls = onto.Single
        rel = rel_cls(base_key)
        if rtitle is not None:
            rel.releaseTitle = str(rtitle).strip()
        if year is not None:
            try:
                rel.releaseYear = int(str(year)[:4])
            except Exception:
                pass
        release_index[base_key] = rel
    if isinstance(rel, onto.Album):
        track.hasAlbum.append(rel)
    else:
        track.hasRelease.append(rel)
    return rel

def _best_track_df(dfs):
    candidates = []
    for name, df in dfs.items():
        cols = {c.lower() for c in df.columns}
        score = 0
        for k in ["track", "song", "title", "artist", "genre", "album", "duration", "year", "id"]:
            score += 1 if any(k in c for c in cols) else 0
        candidates.append((score, name))
    candidates.sort(reverse=True)
    return dfs[candidates[0][1]] if candidates else None

track_df = _best_track_df(dfs)
if track_df is None:
    raise RuntimeError("No CSV files found in /data/musicoset_metadata")

for _, row in track_df.iterrows():
    rowd = row.to_dict()
    t = _get_track(rowd)

    artists = _as_list(_first_present(rowd, ["artist", "artists", "artist_name", "performer", "performers", "main_artist"]))
    if artists:
        for a in artists:
            akey = f"artist_{_slug(a)}"
            ar = _get_or_make(onto.Artist, artist_index, akey, name_prop="artistName", name_value=a)
            t.hasArtist.append(ar)

    performers = _as_list(_first_present(rowd, ["performer", "performers", "performed_by"]))
    if performers:
        for p in performers:
            pkey = f"performer_{_slug(p)}"
            pr = _get_or_make(onto.Performer, artist_index, pkey, name_prop="artistName", name_value=p)
            t.hasPerformer.append(pr)

    composers = _as_list(_first_present(rowd, ["composer", "composers", "written_by"]))
    if composers:
        for c in composers:
            ckey = f"composer_{_slug(c)}"
            co = _get_or_make(onto.Composer, artist_index, ckey, name_prop="entityName", name_value=c)
            t.hasComposer.append(co)

    genres = _as_list(_first_present(rowd, ["genre", "genres", "style"]))
    if genres:
        _attach_many(t, "hasGenre", onto.Genre, genre_index, genres, "genreName")

    subgenres = _as_list(_first_present(rowd, ["subgenre", "sub_genre", "subgenres"]))
    if subgenres:
        _attach_many(t, "hasSubGenre", onto.SubGenre, subgenre_index, subgenres, "genreName")

    instruments = _as_list(_first_present(rowd, ["instrument", "instruments"]))
    if instruments:
        _attach_many(t, "hasInstrument", onto.Instrument, instrument_index, instruments, "instrumentName")

    languages = _as_list(_first_present(rowd, ["language", "languages", "lang"]))
    if languages:
        _attach_many(t, "hasLanguage", onto.Language, language_index, languages, "languageName")

    moods = _as_list(_first_present(rowd, ["mood", "moods", "tag", "tags"]))
    if moods:
        _attach_many(t, "hasMood", onto.Mood, mood_index, moods, "moodName")

    collections = _as_list(_first_present(rowd, ["collection", "archive_collection", "series"]))
    if collections:
        for c in collections:
            ckey = f"collection_{_slug(c)}"
            col = _get_or_make(onto.Collection, collection_index, ckey, name_prop="collectionName", name_value=c)
            t.inCollection.append(col)

    rel = _ensure_release_for_track(t, rowd)
    if rel is not None:
        labels = _as_list(_first_present(rowd, ["label", "record_label", "publisher"]))
        if labels:
            for lab in labels:
                lkey = f"label_{_slug(lab)}"
                lb = _get_or_make(onto.Label, label_index, lkey, name_prop="labelName", name_value=lab)
                rel.hasLabel.append(lb)

for name, df in dfs.items():
    cols = {c.lower() for c in df.columns}
    if any("country" in c for c in cols) and any("artist" in c for c in cols):
        for _, row in df.iterrows():
            rowd = row.to_dict()
            a = _first_present(rowd, ["artist", "artist_name", "artists"])
            c = _first_present(rowd, ["country", "artist_country", "nationality"])
            if a is None or c is None:
                continue
            akey = f"artist_{_slug(a)}"
            ar = _get_or_make(onto.Artist, artist_index, akey, name_prop="artistName", name_value=a)
            ckey = f"country_{_slug(c)}"
            co = _get_or_make(onto.Country, country_index, ckey, name_prop="entityName", name_value=c)
            ar.hasCountry.append(co)

In [7]:
#write simple queries HINT (onto.search, ..)
all_tracks = list(onto.Track.instances())
all_artists = list(onto.Artist.instances())
all_genres = list(onto.Genre.instances())

some_genre = onto.search_one(type=onto.Genre)
tracks_in_some_genre = onto.search(type=onto.Track, hasGenre=some_genre) if some_genre is not None else []

tracks_with_moods = [t for t in onto.Track.instances() if len(getattr(t, "hasMood", [])) > 0]

tracks_by_artist_name_contains_jazz = [
    t for t in onto.Track.instances()
    if any("jazz" in (getattr(a, "artistName", "") or "").lower() for a in getattr(t, "hasArtist", []))
]

all_tracks, all_artists, all_genres, tracks_in_some_genre, tracks_with_moods, tracks_by_artist_name_contains_jazz

([dma_musicoset.track_track_unknown], [], [], [], [], [])

In [8]:
#save
onto.save(file="musicoset_asserted.owl", format="rdfxml")

## OWL reasoning 

Let's look at how reasoning works.

Owlready automatically gets the results of the reasoning from HermiT (a type of reasoner) and reclassifies Individuals and Classes. 

**Exercise 2**
1. think about which things are inferred from your OWL semantics. Query/look at your graph: do you see what you expected?
2. looking at the following tutorial [owlready2-reasoning](https://owlready2.readthedocs.io/en/latest/reasoning.html), which things have not yet been inferred? Run the owlready2 reasoner to:
    - infer these new triples
    - check your ontology and statements (individuals + attributes) for consistency
3. save your asserted + inferred triples to a new file 

In [9]:
#1
onto = get_ontology("musicoset_asserted.owl").load()

first_track = next(iter(onto.Track.instances()), None)
pre_reasoning = {}
pre_reasoning["first_track"] = first_track
pre_reasoning["first_track_artists"] = list(getattr(first_track, "hasArtist", [])) if first_track is not None else []
pre_reasoning["track_parents"] = list(onto.get_parents_of(onto.Track))
pre_reasoning

{'first_track': dma_musicoset.track_track_unknown,
 'first_track_artists': [],
 'track_parents': [owl.Thing,
  dma_musicoset.hasArtist.some(dma_musicoset.Artist),
  dma_musicoset.hasGenre.some(dma_musicoset.Genre),
  dma_musicoset.trackTitle.exactly(1, <class 'str'>)]}

In [10]:
#2
try:
    sync_reasoner()
except Exception:
    sync_reasoner_pellet(infer_property_values=True, infer_data_property_values=True)

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /Users/bara/opt/anaconda3/envs/knowgraphs/lib/python3.10/site-packages/owlready2/hermit:/Users/bara/opt/anaconda3/envs/knowgraphs/lib/python3.10/site-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////var/folders/jj/fclq_2jd0v70f5tsxd4lk3t00000gn/T/tmptcs8v5vh
* Owlready2 * HermiT took 0.2993941307067871 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


In [11]:
#3
post_reasoning = {}
post_reasoning["track_parents"] = list(onto.get_parents_of(onto.Track))
post_reasoning["genre_children"] = list(onto.get_children_of(onto.Genre))
post_reasoning["instances_of_artist"] = list(onto.get_instances_of(onto.Artist))
post_reasoning["instances_of_track"] = list(onto.get_instances_of(onto.Track))

onto.save(file="musicoset_asserted_plus_inferred.owl", format="rdfxml")

post_reasoning

{'track_parents': [owl.Thing,
  dma_musicoset.hasArtist.some(dma_musicoset.Artist),
  dma_musicoset.hasGenre.some(dma_musicoset.Genre),
  dma_musicoset.trackTitle.exactly(1, <class 'str'>)],
 'genre_children': [dma_musicoset.SubGenre],
 'instances_of_artist': [],
 'instances_of_track': [dma_musicoset.track_track_unknown]}

#### Querying inferred triples

**Exercise 3**
Query your inferred triples: 

- *.get_parents_of(entity)* accepts any entity (Class, property or individual), and returns the superclasses (for a class), the superproperties (for a property), or the classes (for an individual). 

- *.get_instances_of(Class)* returns the individuals that are asserted as belonging to the given Class in the ontology. (NB for obtaining all instances, independently of the ontology they are asserted in, use Class.instances()).

- *.get_children_of(entity)* returns the subclasses (or subproperties) that are asserted for the given Class or property in the ontology. (NB for obtaining all children, independently of the ontology they are asserted in, use entity.subclasses()).

In [12]:
onto = get_ontology("musicoset_asserted_plus_inferred.owl").load()

some_track = next(iter(onto.Track.instances()), None)
some_artist = next(iter(onto.Artist.instances()), None)
some_genre = next(iter(onto.Genre.instances()), None)

parents_of_track_class = list(onto.get_parents_of(onto.Track))
children_of_genre_class = list(onto.get_children_of(onto.Genre))

instances_of_track_via_get = list(onto.get_instances_of(onto.Track))
instances_of_track_all = list(onto.Track.instances())

classes_of_some_track = list(onto.get_parents_of(some_track)) if some_track is not None else []

superproperties_of_hasGenre = list(onto.get_parents_of(onto.hasGenre))
subproperties_of_hasArtist = list(onto.get_children_of(onto.hasArtist))

tracks_with_any_mood = [t for t in onto.Track.instances() if len(getattr(t, "hasMood", [])) > 0]
tracks_with_any_instrument = [t for t in onto.Track.instances() if len(getattr(t, "hasInstrument", [])) > 0]

tracks_per_genre = {}
if some_genre is not None:
    tracks_per_genre[str(getattr(some_genre, "genreName", some_genre.name))] = list(onto.search(type=onto.Track, hasGenre=some_genre))

tracks_by_artist_contains = [
    t for t in onto.Track.instances()
    if any("jazz" in (getattr(a, "artistName", "") or "").lower() for a in getattr(t, "hasArtist", []))
]

results_ex3 = {
    "some_track": some_track,
    "some_artist": some_artist,
    "some_genre": some_genre,
    "parents_of_track_class": parents_of_track_class,
    "children_of_genre_class": children_of_genre_class,
    "instances_of_track_via_get_instances_of": instances_of_track_via_get,
    "instances_of_track_via_Class_instances": instances_of_track_all,
    "classes_of_some_track": classes_of_some_track,
    "superproperties_of_hasGenre": superproperties_of_hasGenre,
    "subproperties_of_hasArtist": subproperties_of_hasArtist,
    "tracks_with_any_mood": tracks_with_any_mood,
    "tracks_with_any_instrument": tracks_with_any_instrument,
    "tracks_per_genre_example": tracks_per_genre,
    "tracks_by_artist_name_contains_jazz": tracks_by_artist_contains
}

results_ex3

{'some_track': dma_musicoset.track_track_unknown,
 'some_artist': None,
 'some_genre': None,
 'parents_of_track_class': [owl.Thing,
  dma_musicoset.hasArtist.some(dma_musicoset.Artist),
  dma_musicoset.hasGenre.some(dma_musicoset.Genre),
  dma_musicoset.trackTitle.exactly(1, <class 'str'>)],
 'children_of_genre_class': [dma_musicoset.SubGenre],
 'instances_of_track_via_get_instances_of': [dma_musicoset.track_track_unknown],
 'instances_of_track_via_Class_instances': [dma_musicoset.track_track_unknown],
 'classes_of_some_track': [dma_musicoset.Track],
 'superproperties_of_hasGenre': [],
 'subproperties_of_hasArtist': [],
 'tracks_with_any_mood': [],
 'tracks_with_any_instrument': [],
 'tracks_per_genre_example': {},
 'tracks_by_artist_name_contains_jazz': []}

## Hybrid Intelligence ontology

**Exercise 4**
We can use owlready2 to work with the Hybrid Intelligence (HI) ontology which you will be using for your own project. Using the tools from above, perform the following:
1. Load the HI ontology using owlready2 from the Data folder.
2. Create at least 1 new class with a class restriction.
3. Create at least 1 new object property with an object property restriction.
4. Create at least 1 new data property with a restricted domain and range.


In [13]:
#1
hi_path = "data/hi_ontology.ttl"

if not os.path.exists(hi_path):
    raise FileNotFoundError(f"{hi_path} not found")

hi_onto = get_ontology(f"file://{os.path.abspath(hi_path)}").load(format="turtle")

hi_onto

get_ontology("file:///Users/bara/Desktop/code/KG&ST/KG-tutorials/data/hi_ontology.ttl#")

In [14]:
#2
with hi_onto:
    class TrustedSource(Thing): pass

    class hasTrustScore(DataProperty, FunctionalProperty):
        domain = [TrustedSource]
        range = [float]

    TrustedSource.is_a.append(hasTrustScore.some(float))

In [15]:
#3
with hi_onto:
    class usesSource(ObjectProperty):
        domain = [Thing]
        range = [TrustedSource]

    class EvidenceBackedClaim(Thing): pass

    EvidenceBackedClaim.is_a.append(usesSource.some(TrustedSource))

In [16]:
#4
with hi_onto:
    class confidenceScore(DataProperty, FunctionalProperty):
        domain = [EvidenceBackedClaim]
        range = [float]

In [17]:
#5
with hi_onto:
    src1 = TrustedSource("trusted_source_1")
    src1.hasTrustScore = 0.85

    claim1 = EvidenceBackedClaim("claim_1")
    claim1.usesSource.append(src1)
    claim1.confidenceScore = 0.72

hi_onto.save(file="hi_extended.owl", format="rdfxml")